# AI とチャットする（LINE のような吹き出しで）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/funakoshi-takehiro/library-hiroba/blob/main/notebooks/chat.ipynb)

`ui.chat()` は会話を吹き出しで並べる部品です。自分の言葉は右、AI の言葉は左に出ます。
これに `ai` を組み合わせると、1つのセルで会話ができます。

**このノートブックの内容は、Google Colab でも PyHiroba でも同じように動きます。**

前半はコードから話しかける形、後半は入力欄に打ち込む形です。どちらも両方の環境で
動きますが、**入力欄のほうは 2026-08-09 より前の PyHiroba では動きません**
（そのときは注意書きが出ます）。

> **すぐ確かめたいとき。** 次のセルの `ai.load("auto")` は、この環境に合う
> モデルを選んで読み込みます。GPU の無い Colab では 1.5GB ほど取得するため
> 数分かかります。動作だけを手早く見たい場合は `ai.load("llmjp150m")`
> （255MB）に変えてください。答えは不自然ですが、往復のしくみは同じです。

In [ ]:
%pip install -q -U "library-hiroba[ai]"
# PyHiroba では、このセルの実行は不要です

# このノートブックが前提にしている版。ここより古いと、直したはずの不具合が残る。
# ライブラリを直して公開したら、この数字も上げること
NEEDS = "0.5.4"

import sys
from importlib.metadata import PackageNotFoundError, version


def numbers(text):
    return tuple(int(part) for part in str(text).split(".")[:3])


try:
    installed = version("library-hiroba")
except PackageNotFoundError:
    installed = None  # PyHiroba は同梱なので、配布情報を持ちません
loaded = getattr(sys.modules.get("library_hiroba"), "__version__", None)

if installed and loaded and loaded != installed:
    # 入れ替えても、読み込み済みのものは差し替わらない。再起動するしかない
    print(f"{loaded} が読み込まれたままなので、セッションを再起動します。")
    print(f"再起動したら、このセルから順に実行し直してください（{installed} になります）。")
    try:
        import IPython

        IPython.Application.instance().kernel.do_shutdown(True)
    except Exception:
        print("自動で再起動できませんでした。")
        print("メニューの「ランタイム」→「セッションを再起動する」を実行してください。")
elif installed and numbers(installed) < numbers(NEEDS):
    print(f"このノートブックは {NEEDS} 以降を前提にしていますが、{installed} が入っています。")
    print("このままだと、直したはずの不具合が残ったまま動きます。")
    print("まだ公開されていない版を試すときは、次を実行してから再起動してください:")
    print('  %pip install -q -U "git+https://github.com/funakoshi-takehiro/library-hiroba@main"')
else:
    print("library-hiroba", installed or loaded)

In [ ]:
from library_hiroba import ai, ui

# "auto" は、この環境で実用になるもののうち、いちばん良いモデルを選びます
print(await ai.load("auto"))

## 会話を始める

`ai.talk()` は、AI との会話をひとつ作ります。**前のやりとりを覚えている**ので、
「その高さは？」のように前を受けた聞き方が通じます。

`ai.ask()` が受け取るのは1回分の文章だけで、前に何を話したかは覚えていません。
`ai.talk()` は、その差を埋める後始末を引き受けています。

1. 直前のやりとりを、質問に添えて渡す（記憶）
2. 答えたあとにモデルが自分で書き足した会話の続きを、切り落とす
3. 少しずつ届く答えを、そのつど吹き出しに組み直す

`await ai.load()` は要りません。最初の `ask()` のときに読み込まれます。

In [ ]:
talk = ai.talk()

## 前半：コードから話しかける（Colab・PyHiroba の両方）

`await talk.ask("...")` をセルの最後に置くと、そこまでの会話が吹き出しで出ます。
セルを増やすたびに、話が続きます。

In [ ]:
await talk.ask("日本で一番高い山は？")

In [ ]:
# 「その」が何を指すか、前のやりとりから分かります
await talk.ask("その高さは？")

会話をやり直したいときは `talk.clear()` を呼びます。

`ai.talk(keep=1)` にすると直前の1往復だけを覚えます。小さなモデルは長い文章が苦手なので、
話がかみ合わなくなってきたら減らしてみてください。答えの長さは `max_tokens`、
表示名は `names={"user": "生徒", "assistant": "先生"}` で変えられます。

ためた会話は `talk.messages` で取り出せます（`ui.chat()` にそのまま渡せる形です）。

## 後半：入力欄に打ち込んで話す

`talk.form()` は、入力欄・送信ボタン・吹き出しをまとめて出します。答えは書けたところから
少しずつ吹き出しに足されていきます。

古い PyHiroba で開いた場合だけ、動かない旨の注意書きが上に出ます。

In [ ]:
chat = ai.talk()  # ここまでの会話は引き継がず、新しく始めます
chat.form()

---

確認ポイント:

- `await talk.ask("日本で一番高い山は？")` で吹き出しが2つ出る（あなた／AI）
- 続けて `await talk.ask("その高さは？")` を実行すると、山の話として答える
- 入力欄に打って送信すると、答えが少しずつ書き足されていく

小さなモデルなので、答えが事実と違うことがあります。教材では「AI の答えを確かめる」
題材として使うのが向いています。

### AI を使わないチャット

相手が AI でなくてよければ、`ui.conversation()` だけで作れます。こちらは `ai` を
読み込まないので、待ち時間も通信もありません。

```python
talk = ui.conversation(names={"assistant": "ボット"})
talk.say("こんにちは")
talk.reply("やあ！")
talk
```